# PHASE 8 - Model Comparison

This notebook compares the metrics already produced by Linear Regression, Random Forest, XGBoost, and GRU. It does not retrain any model.

The model is selected using validation RMSE only. The test metrics are then reported as a final held-out comparison and are not used for selection.

In [ ]:
# Nhập các thư viện cần thiết
from pathlib import Path  # Làm việc với đường dẫn file

import matplotlib.pyplot as plt  # Vẽ đồ thị
import pandas as pd  # Xử lý dữ liệu

# Hàm tìm thư mục gốc của project
def find_project_root():
    # Kiểm tra từng vị trí xem có file metrics không
    for candidate in [Path("."), Path("..")]:
        if (candidate / "results/metrics/linear_regression_metrics.csv").exists():
            return candidate
    # Nếu không tìm thấy, báo lỗi
    raise FileNotFoundError("Model metric files were not found in results/metrics.")

# Tìm thư mục gốc
project_root = find_project_root()

# Định nghĩa các đường dẫn thư mục
metrics_dir = project_root / "results/metrics"  # Thư mục lưu metrics
figures_dir = project_root / "results/figures"  # Thư mục lưu hình

# Định nghĩa các file metrics cần tải (từ các phase trước)
metric_files = {
    "Linear Regression": "linear_regression_metrics.csv",  # Từ PHASE 4
    "Random Forest": "random_forest_metrics.csv",  # Từ PHASE 5
    "XGBoost": "xgboost_metrics.csv",  # Từ PHASE 6
    "GRU": "gru_metrics.csv",  # Từ PHASE 7
}

# Danh sách lưu DataFrame của từng model
metrics_frames = []

# Vòng lặp: tải metrics của mỗi model
for model_name, filename in metric_files.items():
    # Tạo đường dẫn file
    path = metrics_dir / filename
    
    # Kiểm tra file tồn tại
    if not path.exists():
        raise FileNotFoundError(f"Missing metrics file: {path}")
    
    # Đọc file CSV
    frame = pd.read_csv(path)
    
    # Thêm cột tên model
    frame["model"] = model_name
    
    # Thêm vào danh sách
    metrics_frames.append(frame)

# Ghép tất cả metrics của các models thành một DataFrame
all_metrics = pd.concat(metrics_frames, ignore_index=True)

# In số lượng models
print(f"Loaded metrics for {all_metrics['model'].nunique()} models.")

## Comparison table

In [ ]:
# === SO SÁNH MÔ HÌNH TRÊN TẬP VALIDATION ===
# Lọc metrics của tập validation, chọn các cột cần thiết, sắp xếp theo RMSE
validation_metrics = (
    all_metrics[all_metrics["split"] == "validation"]  # Lọc chỉ validation data
    [["model", "MAE", "RMSE", "R2", "Training Time"]]  # Chọn các cột cần thiết
    .sort_values("RMSE")  # Sắp xếp theo RMSE (từ nhỏ đến lớn)
    .reset_index(drop=True)  # Reset index (0, 1, 2, ...)
)

# === SO SÁNH MÔ HÌNH TRÊN TẬP TEST ===
# Lọc metrics của tập test
test_metrics = (
    all_metrics[all_metrics["split"] == "test"]  # Lọc chỉ test data
    [["model", "MAE", "RMSE", "R2", "Training Time"]]  # Chọn các cột cần thiết
    .set_index("model")  # Đặt tên model là index
)

# === ĐỊNH DẠNG BẢNG SO SÁNH ===
# Đổi tên các cột thành tiếng Anh đẹp hơn
comparison_columns = {
    "model": "Model",  # model -> Model
    "MAE": "MAE",  # MAE giữ nguyên
    "RMSE": "RMSE",  # RMSE giữ nguyên
    "R2": "R²",  # R2 -> R²
    "Training Time": "Training Time",  # Training Time giữ nguyên
}

# Tạo bảng validation với tên cột mới
validation_comparison = validation_metrics.rename(columns=comparison_columns)

# Tạo bảng test: đảm bảo thứ tự theo validation metrics
# reindex: sắp xếp lại test_metrics theo thứ tự của validation_metrics
# reset_index: chuyển index (model name) thành cột thường
test_comparison = (
    test_metrics.reindex(validation_metrics["model"])  # Sắp xếp theo thứ tự validation
    .reset_index()  # Chuyển index thành cột
    .rename(columns=comparison_columns)  # Đổi tên cột
)

# === HIỂN THỊ VÀ LƯU BẢNG ===
# Hiển thị bảng validation comparison
display(validation_comparison.round(4))  # Làm tròn 4 chữ số thập phân

# Hiển thị bảng test comparison
display(test_comparison.round(4))

# Lưu bảng validation
validation_comparison.to_csv(metrics_dir / "model_comparison_validation.csv", index=False)

# Lưu bảng test
test_comparison.to_csv(metrics_dir / "model_comparison_test.csv", index=False)

# === LỰA CHỌN MÔ HÌNH TỐT NHẤT ===
# Lấy model tốt nhất (hàng đầu tiên, có RMSE nhỏ nhất)
selected_model = validation_comparison.iloc[0]["Model"]

# Lấy metrics test của model được chọn
selected_test = test_comparison[test_comparison["Model"] == selected_model].iloc[0]

# Tạo bảng thông tin model được chọn
selection = pd.DataFrame([{
    "selected_by": "validation RMSE",  # Tiêu chí lựa chọn
    "Model": selected_model,  # Tên model
    "validation_RMSE": validation_comparison.iloc[0]["RMSE"],  # RMSE trên validation
    "test_MAE": selected_test["MAE"],  # MAE trên test (held-out)
    "test_RMSE": selected_test["RMSE"],  # RMSE trên test (held-out)
    "test_R²": selected_test["R²"],  # R² trên test (held-out)
}])

# Lưu thông tin model được chọn
selection.to_csv(metrics_dir / "selected_model.csv", index=False)

# In kết quả
print(f"Selected model by validation RMSE: {selected_model}")
print(f"Held-out test RMSE for selected model: {selected_test['RMSE']:.4f}")

In [ ]:
# === HIỂN THỊ LẠI BẢNG VALIDATION (CELL VỊ HIỂN THỊ) ===
# Lọc metrics của tập validation
validation_metrics = (
    all_metrics[all_metrics["split"] == "validation"]  # Lọc chỉ validation data
    [["model", "MAE", "RMSE", "R2", "Training Time"]]  # Chọn các cột cần thiết
    .sort_values("RMSE")  # Sắp xếp theo RMSE (từ nhỏ đến lớn)
    .reset_index(drop=True)  # Reset index
)

# Đổi tên cột
comparison_columns = {
    "model": "Model",
    "MAE": "MAE",
    "RMSE": "RMSE",
    "R2": "R²",
    "Training Time": "Training Time",
}

# Tạo bảng với tên cột mới
validation_comparison = validation_metrics.rename(columns=comparison_columns)

# Hiển thị bảng
display(validation_comparison.round(4))

# Lưu bảng
validation_comparison.to_csv(metrics_dir / "model_comparison_validation.csv", index=False)

In [ ]:
# === VẼ BIỂU ĐỒ SO SÁNH SAI SỐ ===
# Chuẩn bị dữ liệu: lấy MAE và RMSE từ test_comparison
plot_data = test_comparison.set_index("Model")[["MAE", "RMSE"]]  # Chuyển Model thành index

# Vẽ biểu đồ cột (bar chart)
# kind="bar": loại biểu đồ là cột
# figsize=(11, 5): kích thước 11x5 inch
ax = plot_data.plot(kind="bar", figsize=(11, 5))

# Đặt tiêu đề biểu đồ
# Lưu ý: thứ tự các model là theo validation RMSE (tốt nhất đến kém nhất)
ax.set_title("Test-set error comparison (ordered by validation RMSE)")

# Đặt nhãn trục
ax.set_ylabel("Error")  # Trục Y: giá trị sai số
ax.set_xlabel("Model")  # Trục X: tên model

# Quay nhãn trục X ngang (không chéo)
plt.xticks(rotation=0)

# Điều chỉnh layout
plt.tight_layout()

# Lưu biểu đồ
plt.savefig(figures_dir / "model_comparison_test_errors.png", dpi=150)

# Hiển thị biểu đồ
plt.show()

## Phase 8 conclusion

The comparison is based on saved test metrics, not assumptions. PHASE 9 can analyze the selected model's errors by hour, weekday, month, season, and working-day status.